### Cloning the repo from GitHub

In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

%cd /content
if not os.path.exists('/content/3D_Perception'):
    !git clone https://github.com/giacomosalvatori03/3D_Perception.git
%cd /content/3D_Perception
!git pull
!git fetch origin
!git checkout project2-fusion
!git pull origin project2-fusion

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/3D_Perception
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 430 bytes | 430.00 KiB/s, done.
From https://github.com/giacomosalvatori03/3D_Perception
   f9c990f..e3c5d76  project2-fusion -> origin/project2-fusion
Updating f9c990f..e3c5d76
Fast-forward
 validate_camera.py | 17 +++++++----------
 1 file changed, 7 insertions(+), 10 deletions(-)
Already on 'project2-fusion'
Your branch is up to date with 'origin/project2-fusion'.
From https://github.com/giacomosalvatori03/3D_Perception
 * branch            project2-fusion -> FETCH_HEAD
Already up to date.


In [1]:
import sys

!pip install ultralytics huggingface_hub -q

sys.path.insert(0, "/content/3D_Perception")
from src import CameraDetector

# Initialize the 2D detector with KITTI weights
detector_2d = CameraDetector(conf_thresh=0.25)

# Test inference on a validation frame
sample_img_path = (
    "/content/drive/MyDrive/3D_Perception/data/kitti_validation/image_2/000008.png"
)
detections_2d = detector_2d.detect(sample_img_path)

print(f"\n Found {len(detections_2d)} 2D detections in the image:")
for det in detections_2d:
    bbox = [round(v, 1) for v in det["bbox_2d"]]
    print(
        f"  - [{det['type']}] Confidence: {det['score']:.2f} | 2D BBox: {bbox}"
    )

 Downloading/Loading KITTI YOLOv8 model from HF: dronefreak/kitti-yolov8m/best.pt...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


 Loaded KITTI 2D Detector. Registered classes: {0: 'Car', 1: 'Cyclist', 2: 'Misc', 3: 'Pedestrian', 4: 'Person_sitting', 5: 'Tram', 6: 'Truck', 7: 'Van'}

 Found 9 2D detections in the image:
  - [Car] Confidence: 0.92 | 2D BBox: [884.5, 177.8, 954.4, 241.1]
  - [Car] Confidence: 0.89 | 2D BBox: [738.2, 168.2, 788.1, 209.1]
  - [Car] Confidence: 0.88 | 2D BBox: [594.4, 175.5, 719.0, 264.5]
  - [Car] Confidence: 0.86 | 2D BBox: [945.7, 200.9, 1242.0, 375.0]
  - [Car] Confidence: 0.83 | 2D BBox: [335.7, 173.7, 618.6, 366.1]
  - [Car] Confidence: 0.83 | 2D BBox: [861.7, 173.9, 887.3, 195.8]
  - [Car] Confidence: 0.81 | 2D BBox: [0.1, 174.4, 400.3, 374.6]
  - [Car] Confidence: 0.72 | 2D BBox: [765.9, 170.9, 803.8, 201.6]
  - [Car] Confidence: 0.30 | 2D BBox: [879.2, 181.7, 936.1, 228.3]


In [2]:
!pip install ultralytics huggingface_hub -q

import sys
sys.path.insert(0, "/content/3D_Perception")

from validate_camera import CameraValidator

# Initialize the validation class
validator = CameraValidator(
    data_path="/content/drive/MyDrive/3D_Perception/data/kitti_validation",
    save_dir="/content/drive/MyDrive/3D_Perception/experiments_camera",
    conf_thresh=0.25,
)

# Run validation on all samples (or on max_samples for quick tests)
validator.validate(exp_name="yolov8m_baseline_2d", max_samples=500)

 Downloading/Loading KITTI YOLOv8 model from HF: dronefreak/kitti-yolov8m/best.pt...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


 Loaded KITTI 2D Detector. Registered classes: {0: 'Car', 1: 'Cyclist', 2: 'Misc', 3: 'Pedestrian', 4: 'Person_sitting', 5: 'Tram', 6: 'Truck', 7: 'Van'}

 Starting 2D Camera Validation [YOLOV8M_BASELINE_2D] | Samples: 500/3769
 Output destination: /content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d


Processing 2D Frames: 100%|██████████| 500/500 [15:16<00:00,  1.83s/it]


 2D Validation completed successfully!
  • Predictions saved in: /content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d/pred_labels
  • Ground Truths saved in: /content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d/gt_labels


In [4]:
# 1. Installazione libreria per le metriche di Detection 2D
!pip install torchmetrics pycocotools -q

import os
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

# Mappatura classi -> ID numerico per torchmetrics
CLASS_TO_ID = {"Car": 0, "Pedestrian": 1, "Cyclist": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}


def parse_kitti_label_file(filepath: str, is_pred: bool = False):
    """Legge un file .txt in formato KITTI e ne estrae bbox_2d, score e classe."""
    boxes, scores, labels = [], [], []

    if not os.path.exists(filepath):
        return boxes, scores, labels

    with open(filepath, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue

            cls_name = parts[0]
            if cls_name not in CLASS_TO_ID:
                continue

            # In KITTI 2D: xmin=parts[4], ymin=parts[5], xmax=parts[6], ymax=parts[7]
            bbox_2d = [
                float(parts[4]),
                float(parts[5]),
                float(parts[6]),
                float(parts[7]),
            ]
            score = (
                float(parts[14]) if is_pred and len(parts) >= 15 else 1.0
            )  # Per GT score = 1.0

            boxes.append(bbox_2d)
            scores.append(score)
            labels.append(CLASS_TO_ID[cls_name])

    return boxes, scores, labels


def evaluate_yolo_2d(
    exp_dir: str = "/content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d",
):
    pred_dir = os.path.join(exp_dir, "pred_labels")
    gt_dir = os.path.join(exp_dir, "gt_labels")

    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True)

    pred_files = sorted(
        [f for f in os.listdir(pred_dir) if f.endswith(".txt")]
    )
    print(
        f"📊 Valutazione 2D in corso su {len(pred_files)} frame da: {exp_dir}\n"
    )

    preds_list = []
    target_list = []

    for f_name in tqdm(pred_files, desc="Calcolo metriche 2D"):
        pred_path = os.path.join(pred_dir, f_name)
        gt_path = os.path.join(gt_dir, f_name)

        p_boxes, p_scores, p_labels = parse_kitti_label_file(
            pred_path, is_pred=True
        )
        g_boxes, _, g_labels = parse_kitti_label_file(gt_path, is_pred=False)

        preds_list.append({
            "boxes": torch.tensor(p_boxes, dtype=torch.float32)
            if p_boxes
            else torch.zeros((0, 4)),
            "scores": torch.tensor(p_scores, dtype=torch.float32)
            if p_scores
            else torch.tensor([]),
            "labels": torch.tensor(p_labels, dtype=torch.int64)
            if p_labels
            else torch.tensor([]),
        })

        target_list.append({
            "boxes": torch.tensor(g_boxes, dtype=torch.float32)
            if g_boxes
            else torch.zeros((0, 4)),
            "labels": torch.tensor(g_labels, dtype=torch.int64)
            if g_labels
            else torch.tensor([]),
        })

    metric.update(preds_list, target_list)
    results = metric.compute()

    print("\n" + "=" * 45)
    print("      RISULTATI VALUTAZIONE 2D (YOLOv8)")
    print("=" * 45)
    print(f"  • mAP @ IoU 0.50:0.95 : {results['map'].item()*100:.2f}%")
    print(f"  • mAP @ IoU 0.50      : {results['map_50'].item()*100:.2f}%")
    print(f"  • mAP @ IoU 0.75      : {results['map_75'].item()*100:.2f}%")
    print("-" * 45)

    # Dettaglio per classe (se disponibile)
    if "map_per_class" in results and results["map_per_class"].numel() > 0:
        print("  mAP@50-95 per Classe:")
        for idx, map_val in enumerate(results["map_per_class"]):
            if idx in ID_TO_CLASS:
                val = map_val.item() * 100 if map_val.item() >= 0 else 0.0
                print(f"    - {ID_TO_CLASS[idx]:<10}: {val:.2f}%")
    print("=" * 45)


# Esegui la valutazione
evaluate_yolo_2d()

📊 Valutazione 2D in corso su 500 frame da: /content/drive/MyDrive/3D_Perception/experiments_camera/yolov8m_baseline_2d



Calcolo metriche 2D: 100%|██████████| 500/500 [00:01<00:00, 390.48it/s]



      RISULTATI VALUTAZIONE 2D (YOLOv8)
  • mAP @ IoU 0.50:0.95 : 30.08%
  • mAP @ IoU 0.50      : 48.93%
  • mAP @ IoU 0.75      : 32.46%
---------------------------------------------
  mAP@50-95 per Classe:
    - Car       : 48.25%
    - Pedestrian: 19.73%
    - Cyclist   : 22.26%
